# 📊 Análise Exploratória - Candidaturas Femininas TSE

Este notebook realiza análise exploratória dos dados eleitorais com foco em candidaturas femininas e identificação de oportunidades de marketing político.

## Objetivos:
- Analisar a evolução das candidaturas femininas
- Identificar padrões demográficos e geográficos
- Calcular scores de potencial e diversidade
- Gerar insights para estratégias de marketing

---

In [ ]:
# Imports essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from pathlib import Path
import sys

# Configurações
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Adicionar src ao path
sys.path.append('../')

print("✅ Imports realizados com sucesso!")

In [ ]:
# Configurações do projeto
from src.config import settings
from src.processing.data_processor import DataProcessor

# Paths
SILVER_PATH = Path(settings.SILVER_PATH)
GOLD_PATH = Path(settings.GOLD_PATH)

print(f"📁 Silver Path: {SILVER_PATH}")
print(f"📁 Gold Path: {GOLD_PATH}")
print(f"🗳️ Anos de eleições majoritárias: {settings.ELECTION_YEARS_MAJOR}")
print(f"🏛️ Anos de eleições locais: {settings.ELECTION_YEARS_LOCAL}")

## 1. 📥 Carregamento e Preparação dos Dados

In [ ]:
def load_processed_data():
    """Carrega todos os dados processados da camada Silver"""
    data_files = list(SILVER_PATH.glob("candidatos_*_processed.parquet"))
    
    if not data_files:
        print("❌ Nenhum arquivo de dados processados encontrado!")
        print("Execute primeiro o pipeline de ingestão e processamento.")
        return pd.DataFrame()
    
    print(f"📂 Encontrados {len(data_files)} arquivos de dados:")
    for file in data_files:
        print(f"  - {file.name}")
    
    # Carregar e concatenar todos os dados
    dfs = []
    for file_path in data_files:
        df = pd.read_parquet(file_path)
        print(f"  ✅ {file_path.name}: {len(df):,} registros")
        dfs.append(df)
    
    combined_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"\n📊 Total combinado: {len(combined_df):,} candidatos")
    
    return combined_df

# Carregar dados
df_all = load_processed_data()

In [ ]:
# Verificar estrutura dos dados
if not df_all.empty:
    print("📋 Informações gerais dos dados:")
    print(f"  - Shape: {df_all.shape}")
    print(f"  - Colunas: {len(df_all.columns)}")
    print(f"  - Memória: {df_all.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    print("\n📊 Primeiras colunas:")
    display(df_all.head())
    
    print("\n📈 Informações dos dados:")
    display(df_all.info())
else:
    print("❌ Dados vazios! Execute o pipeline de processamento primeiro.")

## 2. 👩‍💼 Análise Específica de Candidaturas Femininas

In [ ]:
# Filtrar dados de mulheres
df_women = df_all[df_all['IS_WOMAN'] == True].copy() if not df_all.empty else pd.DataFrame()

if not df_women.empty:
    print(f"👩‍💼 Total de candidatas: {len(df_women):,}")
    print(f"📊 Percentual do total: {(len(df_women) / len(df_all)) * 100:.1f}%")
    
    # Estatísticas básicas
    print("\n📈 Distribuição por ano:")
    if 'ANO_ELEICAO' in df_women.columns:
        year_counts = df_women['ANO_ELEICAO'].value_counts().sort_index()
        for year, count in year_counts.items():
            print(f"  {year}: {count:,} candidatas")
    
    print("\n🌍 Distribuição por região:")
    if 'REGIAO' in df_women.columns:
        region_counts = df_women['REGIAO'].value_counts()
        for region, count in region_counts.items():
            print(f"  {region}: {count:,} candidatas")
else:
    print("❌ Nenhuma candidata encontrada nos dados!")

In [ ]:
# Visualização: Evolução temporal das candidaturas femininas
if not df_women.empty and 'ANO_ELEICAO' in df_women.columns:
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Evolução do Número de Candidatas',
            'Candidatas por Região',
            'Distribuição por Cor/Raça',
            'Distribuição por Categoria de Cargo'
        ],
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"type": "pie"}, {"secondary_y": False}]]
    )
    
    # 1. Evolução temporal
    yearly_counts = df_women.groupby('ANO_ELEICAO').size().reset_index(name='count')
    fig.add_trace(
        go.Scatter(
            x=yearly_counts['ANO_ELEICAO'],
            y=yearly_counts['count'],
            mode='lines+markers',
            name='Candidatas',
            line=dict(color='#ff6b9d', width=3),
            marker=dict(size=8)
        ),
        row=1, col=1
    )
    
    # 2. Por região
    if 'REGIAO' in df_women.columns:
        region_counts = df_women['REGIAO'].value_counts()
        fig.add_trace(
            go.Bar(
                x=region_counts.index,
                y=region_counts.values,
                name='Por Região',
                marker_color='#74b9ff'
            ),
            row=1, col=2
        )
    
    # 3. Por cor/raça
    if 'COR_RACA' in df_women.columns:
        race_counts = df_women['COR_RACA'].value_counts()
        fig.add_trace(
            go.Pie(
                labels=race_counts.index,
                values=race_counts.values,
                name="Cor/Raça"
            ),
            row=2, col=1
        )
    
    # 4. Por categoria de cargo
    if 'CARGO_CATEGORY' in df_women.columns:
        cargo_counts = df_women['CARGO_CATEGORY'].value_counts().head(6)
        fig.add_trace(
            go.Bar(
                x=cargo_counts.values,
                y=cargo_counts.index,
                orientation='h',
                name='Por Cargo',
                marker_color='#00b894'
            ),
            row=2, col=2
        )
    
    fig.update_layout(
        height=800,
        title_text="Análise Multidimensional - Candidaturas Femininas",
        showlegend=False
    )
    
    fig.show()

## 3. 🎯 Análise de Potencial e Scores

In [ ]:
# Análise dos scores de potencial
if not df_women.empty:
    score_columns = ['WOMEN_POTENTIAL_SCORE', 'MARKETING_POTENTIAL', 'DIVERSITY_SCORE']
    available_scores = [col for col in score_columns if col in df_women.columns]
    
    if available_scores:
        print("📊 Estatísticas dos Scores:")
        print(df_women[available_scores].describe())
        
        # Visualização dos scores
        fig, axes = plt.subplots(1, len(available_scores), figsize=(15, 5))
        
        if len(available_scores) == 1:
            axes = [axes]
        
        for i, score in enumerate(available_scores):
            sns.histplot(data=df_women, x=score, bins=20, kde=True, ax=axes[i])
            axes[i].set_title(f'Distribuição: {score.replace("_", " ").title()}')
            axes[i].axvline(df_women[score].mean(), color='red', linestyle='--', 
                          label=f'Média: {df_women[score].mean():.3f}')
            axes[i].legend()
        
        plt.tight_layout()
        plt.show()
    else:
        print("❌ Colunas de scores não encontradas nos dados!")

In [ ]:
# Identificar candidatas com alto potencial
if not df_women.empty and 'WOMEN_POTENTIAL_SCORE' in df_women.columns:
    # Definir thresholds
    high_potential_threshold = 0.7
    very_high_potential_threshold = 0.8
    
    high_potential_women = df_women[df_women['WOMEN_POTENTIAL_SCORE'] >= high_potential_threshold]
    very_high_potential_women = df_women[df_women['WOMEN_POTENTIAL_SCORE'] >= very_high_potential_threshold]
    
    print(f"🌟 Candidatas com alto potencial (>= {high_potential_threshold}): {len(high_potential_women):,}")
    print(f"⭐ Candidatas com potencial muito alto (>= {very_high_potential_threshold}): {len(very_high_potential_women):,}")
    
    if len(high_potential_women) > 0:
        print("\n🏆 Top 10 candidatas com maior potencial:")
        top_candidates = high_potential_women.nlargest(10, 'WOMEN_POTENTIAL_SCORE')
        
        display_cols = ['NM_CANDIDATO', 'NM_UE', 'REGIAO', 'CARGO_CATEGORY', 
                       'WOMEN_POTENTIAL_SCORE', 'MARKETING_POTENTIAL']
        available_display_cols = [col for col in display_cols if col in top_candidates.columns]
        
        display(top_candidates[available_display_cols])

## 4. 🌍 Análise Geográfica e Regional

In [ ]:
# Análise por região
if not df_women.empty and 'REGIAO' in df_women.columns:
    print("🌍 Análise Regional das Candidaturas Femininas:")
    
    # Estatísticas por região
    regional_stats = df_women.groupby('REGIAO').agg({
        'NM_CANDIDATO': 'count',
        'WOMEN_POTENTIAL_SCORE': ['mean', 'std'] if 'WOMEN_POTENTIAL_SCORE' in df_women.columns else 'count',
        'MARKETING_POTENTIAL': ['mean', 'std'] if 'MARKETING_POTENTIAL' in df_women.columns else 'count',
        'DIVERSITY_SCORE': ['mean', 'std'] if 'DIVERSITY_SCORE' in df_women.columns else 'count'
    }).round(3)
    
    regional_stats.columns = ['Total_Candidatas', 'Pot_Medio', 'Pot_StdDev', 
                             'Mark_Medio', 'Mark_StdDev', 'Div_Medio', 'Div_StdDev']
    
    display(regional_stats)
    
    # Visualização regional
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. Número de candidatas por região
    region_counts = df_women['REGIAO'].value_counts()
    region_counts.plot(kind='bar', ax=axes[0,0], color='skyblue')
    axes[0,0].set_title('Candidatas por Região')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. Potencial médio por região
    if 'WOMEN_POTENTIAL_SCORE' in df_women.columns:
        potential_by_region = df_women.groupby('REGIAO')['WOMEN_POTENTIAL_SCORE'].mean()
        potential_by_region.plot(kind='bar', ax=axes[0,1], color='lightcoral')
        axes[0,1].set_title('Potencial Médio por Região')
        axes[0,1].tick_params(axis='x', rotation=45)
    
    # 3. Diversidade por região
    if 'DIVERSITY_SCORE' in df_women.columns:
        diversity_by_region = df_women.groupby('REGIAO')['DIVERSITY_SCORE'].mean()
        diversity_by_region.plot(kind='bar', ax=axes[1,0], color='lightgreen')
        axes[1,0].set_title('Diversidade Média por Região')
        axes[1,0].tick_params(axis='x', rotation=45)
    
    # 4. Marketing potential por região
    if 'MARKETING_POTENTIAL' in df_women.columns:
        marketing_by_region = df_women.groupby('REGIAO')['MARKETING_POTENTIAL'].mean()
        marketing_by_region.plot(kind='bar', ax=axes[1,1], color='gold')
        axes[1,1].set_title('Potencial de Marketing por Região')
        axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 5. 🎨 Análise de Diversidade

In [ ]:
# Análise de diversidade racial e de gênero
if not df_all.empty:
    print("🎨 Análise de Diversidade:")
    
    # Distribuição por gênero
    if 'GENERO' in df_all.columns:
        gender_counts = df_all['GENERO'].value_counts()
        print(f"\n👥 Distribuição por Gênero:")
        for gender, count in gender_counts.items():
            percentage = (count / len(df_all)) * 100
            print(f"  {gender}: {count:,} ({percentage:.1f}%)")
    
    # Distribuição racial entre mulheres
    if not df_women.empty and 'COR_RACA' in df_women.columns:
        race_counts_women = df_women['COR_RACA'].value_counts()
        print(f"\n🌈 Distribuição Racial - Candidatas:")
        for race, count in race_counts_women.items():
            percentage = (count / len(df_women)) * 100
            print(f"  {race}: {count:,} ({percentage:.1f}%)")
    
    # Interseccionalidade: Mulheres + Raça
    if not df_women.empty and 'IS_MINORITY_RACE' in df_women.columns:
        minority_women = df_women[df_women['IS_MINORITY_RACE'] == True]
        print(f"\n⚡ Interseccionalidade:")
        print(f"  Mulheres de minorias raciais: {len(minority_women):,}")
        print(f"  Percentual das candidatas: {(len(minority_women) / len(df_women)) * 100:.1f}%")
        print(f"  Percentual do total: {(len(minority_women) / len(df_all)) * 100:.1f}%")

In [ ]:
# Visualização de diversidade
if not df_all.empty:
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            'Distribuição por Gênero (Geral)',
            'Distribuição Racial (Candidatas)',
            'Interseccionalidade: Mulheres por Raça',
            'Score de Diversidade - Distribuição'
        ],
        specs=[[{"type": "pie"}, {"type": "pie"}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # 1. Distribuição por gênero (geral)
    if 'GENERO' in df_all.columns:
        gender_counts = df_all['GENERO'].value_counts()
        fig.add_trace(
            go.Pie(
                labels=gender_counts.index,
                values=gender_counts.values,
                name="Gênero Geral",
                marker_colors=['#ff6b9d', '#74b9ff']
            ),
            row=1, col=1
        )
    
    # 2. Distribuição racial (candidatas)
    if not df_women.empty and 'COR_RACA' in df_women.columns:
        race_counts_women = df_women['COR_RACA'].value_counts()
        fig.add_trace(
            go.Pie(
                labels=race_counts_women.index,
                values=race_counts_women.values,
                name="Raça Candidatas"
            ),
            row=1, col=2
        )
    
    # 3. Interseccionalidade por região
    if not df_women.empty and 'REGIAO' in df_women.columns and 'IS_MINORITY_RACE' in df_women.columns:
        intersect_data = df_women.groupby(['REGIAO', 'IS_MINORITY_RACE']).size().unstack(fill_value=0)
        
        fig.add_trace(
            go.Bar(
                x=intersect_data.index,
                y=intersect_data[True] if True in intersect_data.columns else [],
                name='Minoritárias',
                marker_color='#e17055'
            ),
            row=2, col=1
        )
        
        fig.add_trace(
            go.Bar(
                x=intersect_data.index,
                y=intersect_data[False] if False in intersect_data.columns else [],
                name='Não Minoritárias',
                marker_color='#74b9ff'
            ),
            row=2, col=1
        )
    
    # 4. Score de diversidade
    if 'DIVERSITY_SCORE' in df_all.columns:
        diversity_scores = df_all['DIVERSITY_SCORE'].dropna()
        if len(diversity_scores) > 0:
            fig.add_trace(
                go.Histogram(
                    x=diversity_scores,
                    nbinsx=20,
                    name='Diversity Score',
                    marker_color='#00b894'
                ),
                row=2, col=2
            )
    
    fig.update_layout(
        height=800,
        title_text="Análise Completa de Diversidade",
        showlegend=True
    )
    
    fig.show()

## 6. 💡 Insights e Recomendações para Marketing

In [ ]:
# Gerar insights automatizados
def generate_insights(df_all, df_women):
    """Gera insights automatizados baseados nos dados"""
    insights = []
    
    if not df_all.empty and not df_women.empty:
        
        # 1. Representatividade feminina
        women_percentage = (len(df_women) / len(df_all)) * 100
        insights.append(f"👩‍💼 Representatividade: {women_percentage:.1f}% dos candidatos são mulheres")
        
        # 2. Região com maior potencial
        if 'REGIAO' in df_women.columns and 'MARKETING_POTENTIAL' in df_women.columns:
            region_potential = df_women.groupby('REGIAO')['MARKETING_POTENTIAL'].mean().sort_values(ascending=False)
            if not region_potential.empty:
                top_region = region_potential.index[0]
                top_score = region_potential.iloc[0]
                insights.append(f"🌟 Maior potencial: Região {top_region} (score: {top_score:.3f})")
        
        # 3. Diversidade racial
        if 'IS_MINORITY_RACE' in df_women.columns:
            minority_women = len(df_women[df_women['IS_MINORITY_RACE'] == True])
            minority_percentage = (minority_women / len(df_women)) * 100
            insights.append(f"🌈 Diversidade: {minority_percentage:.1f}% das candidatas são de minorias raciais")
        
        # 4. Candidatas de alto potencial
        if 'WOMEN_POTENTIAL_SCORE' in df_women.columns:
            high_potential = len(df_women[df_women['WOMEN_POTENTIAL_SCORE'] > 0.7])
            high_potential_percentage = (high_potential / len(df_women)) * 100
            insights.append(f"⭐ Alto potencial: {high_potential} candidatas ({high_potential_percentage:.1f}%) têm score > 0.7")
        
        # 5. Tendência temporal
        if 'ANO_ELEICAO' in df_women.columns:
            yearly_counts = df_women.groupby('ANO_ELEICAO').size()
            if len(yearly_counts) > 1:
                growth = yearly_counts.iloc[-1] - yearly_counts.iloc[0]
                trend = "crescimento" if growth > 0 else "decréscimo"
                insights.append(f"📈 Tendência: {trend} de {abs(growth)} candidatas entre {yearly_counts.index[0]} e {yearly_counts.index[-1]}")
        
        # 6. Cargo com maior diversidade
        if 'CARGO_CATEGORY' in df_women.columns and 'DIVERSITY_SCORE' in df_women.columns:
            cargo_diversity = df_women.groupby('CARGO_CATEGORY')['DIVERSITY_SCORE'].mean().sort_values(ascending=False)
            if not cargo_diversity.empty:
                diverse_cargo = cargo_diversity.index[0]
                diverse_score = cargo_diversity.iloc[0]
                insights.append(f"🏛️ Cargo mais diverso: {diverse_cargo} (score: {diverse_score:.3f})")
    
    return insights

# Gerar e exibir insights
insights = generate_insights(df_all, df_women)

print("💡 INSIGHTS PARA ESTRATÉGIAS DE MARKETING:")
print("=" * 50)

for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\n🎯 RECOMENDAÇÕES ESTRATÉGICAS:")
print("=" * 50)

recommendations = [
    "📍 Foque em regiões com maior potencial de marketing identificadas",
    "🌈 Priorize candidatas de minorias raciais para maximizar diversidade",
    "⭐ Invista em candidatas com score de potencial > 0.7",
    "📱 Desenvolva estratégias digitais para candidatas jovens",
    "🤝 Crie alianças regionais baseadas nos dados de performance",
    "📊 Monitore continuamente os indicadores de potencial e ajuste estratégias"
]

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

## 7. 📁 Salvamento de Dados para Camada Gold

In [ ]:
# Preparar dados para camada Gold (agregados e prontos para BI)
def prepare_gold_datasets(df_all, df_women):
    """Prepara datasets agregados para a camada Gold"""
    
    gold_datasets = {}
    
    if not df_all.empty:
        
        # 1. Resumo por ano
        if 'ANO_ELEICAO' in df_all.columns:
            yearly_summary = df_all.groupby('ANO_ELEICAO').agg({
                'NM_CANDIDATO': 'count',
                'IS_WOMAN': 'sum',
                'IS_MINORITY_RACE': 'sum' if 'IS_MINORITY_RACE' in df_all.columns else 'count',
                'DIVERSITY_SCORE': 'mean' if 'DIVERSITY_SCORE' in df_all.columns else 'count'
            }).round(3)
            
            yearly_summary.columns = ['Total_Candidatos', 'Total_Mulheres', 'Total_Minorias', 'Diversidade_Media']
            yearly_summary['Percentual_Mulheres'] = (yearly_summary['Total_Mulheres'] / yearly_summary['Total_Candidatos'] * 100).round(1)
            
            gold_datasets['resumo_anual'] = yearly_summary.reset_index()
        
        # 2. Resumo por região
        if 'REGIAO' in df_all.columns:
            regional_summary = df_all.groupby('REGIAO').agg({
                'NM_CANDIDATO': 'count',
                'IS_WOMAN': 'sum',
                'WOMEN_POTENTIAL_SCORE': 'mean' if 'WOMEN_POTENTIAL_SCORE' in df_all.columns else 'count',
                'MARKETING_POTENTIAL': 'mean' if 'MARKETING_POTENTIAL' in df_all.columns else 'count'
            }).round(3)
            
            regional_summary.columns = ['Total_Candidatos', 'Total_Mulheres', 'Potencial_Medio', 'Marketing_Medio']
            gold_datasets['resumo_regional'] = regional_summary.reset_index()
    
    if not df_women.empty:
        
        # 3. Top candidatas por potencial
        if 'WOMEN_POTENTIAL_SCORE' in df_women.columns:
            top_candidates = df_women.nlargest(100, 'WOMEN_POTENTIAL_SCORE')[[
                'NM_CANDIDATO', 'NM_UE', 'REGIAO', 'CARGO_CATEGORY',
                'WOMEN_POTENTIAL_SCORE', 'MARKETING_POTENTIAL', 'DIVERSITY_SCORE',
                'COR_RACA', 'ANO_ELEICAO'
            ]]
            
            gold_datasets['top_candidatas'] = top_candidates
        
        # 4. Análise de diversidade
        if 'COR_RACA' in df_women.columns and 'REGIAO' in df_women.columns:
            diversity_analysis = df_women.groupby(['REGIAO', 'COR_RACA']).size().unstack(fill_value=0)
            gold_datasets['diversidade_regional'] = diversity_analysis.reset_index()
    
    return gold_datasets

# Preparar datasets da camada Gold
gold_data = prepare_gold_datasets(df_all, df_women)

# Salvar na camada Gold
GOLD_PATH.mkdir(parents=True, exist_ok=True)

for dataset_name, dataset in gold_data.items():
    if not dataset.empty:
        file_path = GOLD_PATH / f"{dataset_name}.parquet"
        dataset.to_parquet(file_path, index=False)
        print(f"✅ Salvo: {file_path} ({len(dataset)} registros)")
        
        # Também salvar em CSV para fácil acesso
        csv_path = GOLD_PATH / f"{dataset_name}.csv"
        dataset.to_csv(csv_path, index=False)
        print(f"📊 CSV: {csv_path}")

print(f"\n🏆 Datasets da camada Gold salvos em: {GOLD_PATH}")

## 8. 📊 Relatório Final

In [ ]:
# Gerar relatório final
def generate_final_report(df_all, df_women, gold_data):
    """Gera relatório final da análise"""
    
    report = {
        'timestamp': pd.Timestamp.now().isoformat(),
        'data_overview': {},
        'women_analysis': {},
        'diversity_metrics': {},
        'recommendations': []
    }
    
    if not df_all.empty:
        report['data_overview'] = {
            'total_candidates': len(df_all),
            'total_women': len(df_women) if not df_women.empty else 0,
            'women_percentage': (len(df_women) / len(df_all) * 100) if not df_women.empty else 0,
            'years_covered': sorted(df_all['ANO_ELEICAO'].unique().tolist()) if 'ANO_ELEICAO' in df_all.columns else [],
            'regions_covered': sorted(df_all['REGIAO'].unique().tolist()) if 'REGIAO' in df_all.columns else []
        }
    
    if not df_women.empty:
        report['women_analysis'] = {
            'total_women_candidates': len(df_women),
            'high_potential_candidates': len(df_women[df_women['WOMEN_POTENTIAL_SCORE'] > 0.7]) if 'WOMEN_POTENTIAL_SCORE' in df_women.columns else 0,
            'avg_potential_score': df_women['WOMEN_POTENTIAL_SCORE'].mean() if 'WOMEN_POTENTIAL_SCORE' in df_women.columns else 0,
            'avg_marketing_potential': df_women['MARKETING_POTENTIAL'].mean() if 'MARKETING_POTENTIAL' in df_women.columns else 0,
            'top_region': df_women.groupby('REGIAO')['MARKETING_POTENTIAL'].mean().idxmax() if 'REGIAO' in df_women.columns and 'MARKETING_POTENTIAL' in df_women.columns else 'N/A'
        }
        
        if 'IS_MINORITY_RACE' in df_women.columns:
            report['diversity_metrics'] = {
                'minority_women_count': len(df_women[df_women['IS_MINORITY_RACE'] == True]),
                'minority_women_percentage': (len(df_women[df_women['IS_MINORITY_RACE'] == True]) / len(df_women) * 100),
                'avg_diversity_score': df_women['DIVERSITY_SCORE'].mean() if 'DIVERSITY_SCORE' in df_women.columns else 0
            }
    
    # Recomendações baseadas nos dados
    if not df_women.empty:
        if 'WOMEN_POTENTIAL_SCORE' in df_women.columns:
            high_potential_pct = len(df_women[df_women['WOMEN_POTENTIAL_SCORE'] > 0.7]) / len(df_women) * 100
            if high_potential_pct > 20:
                report['recommendations'].append("Alto potencial identificado: Foque em candidatas com score > 0.7")
            else:
                report['recommendations'].append("Potencial limitado: Invista em desenvolvimento de lideranças")
        
        if 'REGIAO' in df_women.columns and 'MARKETING_POTENTIAL' in df_women.columns:
            top_region = df_women.groupby('REGIAO')['MARKETING_POTENTIAL'].mean().idxmax()
            report['recommendations'].append(f"Priorize investimentos na região {top_region}")
    
    return report

# Gerar relatório
final_report = generate_final_report(df_all, df_women, gold_data)

# Salvar relatório
report_path = GOLD_PATH / "relatorio_final.json"
with open(report_path, 'w', encoding='utf-8') as f:
    import json
    json.dump(final_report, f, indent=2, ensure_ascii=False, default=str)

print("📋 RELATÓRIO FINAL")
print("=" * 50)
print(f"📊 Total de candidatos analisados: {final_report['data_overview'].get('total_candidates', 0):,}")
print(f"👩‍💼 Total de candidatas: {final_report['women_analysis'].get('total_women_candidates', 0):,}")
print(f"⭐ Candidatas de alto potencial: {final_report['women_analysis'].get('high_potential_candidates', 0):,}")
print(f"🎯 Score médio de potencial: {final_report['women_analysis'].get('avg_potential_score', 0):.3f}")
print(f"🌟 Região top: {final_report['women_analysis'].get('top_region', 'N/A')}")

print(f"\n💾 Relatório salvo em: {report_path}")
print(f"📁 Datasets Gold disponíveis em: {GOLD_PATH}")

print("\n✅ Análise concluída com sucesso!")